# [7.1] Logit Lens, Tuned Lens, and Patchscopes - Solutions

Reference validation notebook for the section-local activation-to-language implementation. This executes the visible tests against `solutions.py`, checks the CPU notebook contract, and verifies the committed CUDA report highlights.

Expected CUDA highlights: pinned TransformerLens `gelu-1l` loads on CUDA, a ridge tuned lens improves held-out next-token decoding over ordinary logit lens, Patchscope-style clean activations beat corrupt text-only logits, counterfactual activations change the decoded answer, random activations stay low-confidence, and the attention-lens diagnostic is finite.


In [ ]:
import json
import sys
from pathlib import Path

chapter = "chapter7_activation_to_language"
section = "part1_lenses_patchscopes"
root_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section
if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part1_lenses_patchscopes.tests as tests
from part1_lenses_patchscopes import solutions


In [ ]:
tests.test_logit_lens_and_top_tokens_match_reference(
    solutions.logit_lens,
    solutions.top_tokens,
)
tests.test_tuned_lens_improves_over_logit_lens_on_toy_targets(
    solutions.logit_lens,
    solutions.tuned_lens,
    solutions.lens_accuracy_report,
)
tests.test_attention_lens_decodes_attention_weighted_values(
    solutions.attention_lens,
)
tests.test_patchscope_templates_and_accuracy_report(
    solutions.patchscope_prompt,
    solutions.patchscope_accuracy_report,
)
tests.test_counterfactual_and_random_activation_controls(
    solutions.counterfactual_activation_report,
    solutions.random_activation_confidence_report,
)
tests.test_notebook_contract(solutions.run_smoke_test)


In [ ]:
contract = solutions.run_smoke_test(cpu=True)
contract


In [ ]:
report = json.loads((section_dir / "verification_report.json").read_text())
gpu = report["metrics"]["gpu_test"]
assert gpu["preflight_passed"], "7.1 CUDA preflight should pass."
assert gpu["model_name"] == "gelu-1l", "The real-model path should use pinned TransformerLens gelu-1l."
assert gpu["hf_revision"] == "bddc0e332f0ae84279e6a6a45d91b314899e1603", "gelu-1l revision should remain pinned."
assert gpu["tuned_lens_accuracy"] >= 0.35, "Tuned lens should clear the held-out accuracy floor."
assert gpu["tuned_lens_improvement"] >= 0.30, "Tuned lens should improve over ordinary logit lens."
assert gpu["final_decode_max_abs_error"] <= 1e-4, "Final normalized residual decode should match model logits to tolerance."
assert gpu["patchscope_accuracy"] == 1.0, "Activation-conditioned clean logits should recover clean answers."
assert gpu["text_only_accuracy"] == 0.0, "Corrupt text-only baseline should fail this Patchscope pair set."
assert gpu["patchscope_beats_text_only"], "Patchscope should beat the text-only baseline."
assert gpu["counterfactual_changed"], "Counterfactual residual decode should change the answer."
assert gpu["random_max_confidence"] <= 0.1, "Small random activations should stay low confidence."
assert gpu["attention_lens_finite"], "Real attention-lens diagnostic should be finite."
assert gpu["peak_vram_gb"] <= 1.0, "The 7.1 preflight should stay under the locked 1GB budget."
{
    "logit_lens_accuracy": gpu["logit_lens_accuracy"],
    "tuned_lens_accuracy": gpu["tuned_lens_accuracy"],
    "tuned_lens_improvement": gpu["tuned_lens_improvement"],
    "patchscope_accuracy": gpu["patchscope_accuracy"],
    "random_max_confidence": gpu["random_max_confidence"],
    "peak_vram_gb": gpu["peak_vram_gb"],
}
